In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("=== 05b: Fault Screening Design — Environment Check ===")
print("Python:", sys.version.split()[0])
print("Drive accessible:", Path("/content/drive/MyDrive").is_dir())

for package in [
    "torch", "torchvision", "ultralytics",
    "numpy", "pycocotools",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("CUDA availability: NOT CHECKED")

print("No packages installed. No files modified. No inference performed.")

=== 05b: Fault Screening Design — Environment Check ===
Python: 3.13.18
Drive accessible: False
torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
CUDA available: False
No packages installed. No files modified. No inference performed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
report_path = (
    repo / "docs/research_reports/"
    "05a_shared_INT8_clean_evaluation_v1.json"
)

expected_head = "30661e980b0854e92cb36fde105f5723cb14ef39"
expected_report_sha = (
    "47861220401cc36ea2c6c19f427d351dc"
    "bb5ed39a9067fa79e0abbb175f889d7"
)

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()

head = git("rev-parse", "HEAD")
if head != expected_head:
    raise RuntimeError(f"Unexpected repository HEAD: {head}")

content = report_path.read_bytes()
if hashlib.sha256(content).hexdigest() != expected_report_sha:
    raise RuntimeError("Published evaluation report SHA256 mismatch.")

publication = json.loads(content)
report = publication["evaluation_record"]["payload"]

print("Git HEAD:", head)
print("Working tree:", git("status", "--short") or "(clean)")
print("Published 05a report SHA256: PASS")
print("Quality gate:", report["quality_gate_decision"])

for name, metrics in report["metrics"].items():
    print(name, {
        key: metrics[key]
        for key in ("AP50_95", "AP50", "AP75", "AR100")
    })

if report["quality_gate_decision"] != "PASS":
    raise RuntimeError("The shared clean baseline did not pass its gate.")

print("No files modified. No model loaded. No inference performed.")

Git HEAD: 30661e980b0854e92cb36fde105f5723cb14ef39
Working tree: (clean)
Published 05a report SHA256: PASS
Quality gate: PASS
fp32_clean {'AP50_95': 0.46140445176466793, 'AP50': 0.6202350455059251, 'AP75': 0.5015263049059855, 'AR100': 0.6158806984346427}
int8_channel_clean {'AP50_95': 0.4607655860161343, 'AP50': 0.6196714332240089, 'AP75': 0.5024944464020231, 'AR100': 0.6153007944762907}
No files modified. No model loaded. No inference performed.


In [ ]:
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

paths = [
    "docs/research_reports/Notebook05_Design_Review_Decision_v1.json",
    "docs/research_reports/Notebook05_Layer_Coverage_Audit_v1.json",
    "docs/research_reports/Notebook05_Shared_Baseline_Candidate_Review_v1.json",
]

for relative_path in paths:
    path = repo / relative_path
    print(f"\n--- {relative_path} ---")
    if not path.is_file():
        raise FileNotFoundError(relative_path)
    print(path.read_text(encoding="utf-8"))

print("\nRead-only inspection.")
print("No scenarios generated. No files modified. No inference performed.")


--- docs/research_reports/Notebook05_Design_Review_Decision_v1.json ---
{
  "schema_version": "1.0",
  "recorded_at_utc": "2026-11-11T10:49:00.639427+00:00",
  "status": "INITIAL_CAMPAIGN_PAUSED_PENDING_REDESIGN",
  "initial_plan": {
    "selected_layers": 12,
    "planned_fault_scenarios": 240,
    "completed_summarized_scenarios": 60,
    "remaining_planned_scenarios": 180,
    "development_images_per_scenario": 1000
  },
  "observation_disclosure": "Decision made after observing the pilot and the first 60 scenarios from the first three layers in the frozen execution order.",
  "reasons": [
    "Selected layers do not cover all eligible architectural locations.",
    "Five sampled weights per bit have no validated precision guarantee.",
    "Mean signed AP change can conceal damaging individual scenarios.",
    "Target-layer-only INT4 models differ across layer comparisons.",
    "Final protection selection requires a justified broader design."
  ],
  "interpretation": [
    "Preser

In [ ]:
import json

screening_draft = {
    "status": "DRAFT_NOT_FROZEN",
    "baseline": {
        "source": "05a_shared_INT8_clean_evaluation_v1.json",
        "representation": "logical_signed_INT8_weights_FP32_computation",
        "quantization": "per_output_channel_symmetric",
        "shared_clean_model_across_all_fault_scenarios": True,
    },
    "scope": {
        "eligible_conv2d_layers": 63,
        "locations": "Conv2d weight elements only",
        "excluded": [
            "fixed DFL projection",
            "biases",
            "quantization scales",
            "activations",
            "other parameters and buffers",
        ],
        "claim": "logical weight-fault sensitivity within this scope",
    },
    "fault_model": {
        "operation": "single_bit_XOR_in_8bit_twos_complement",
        "bits": list(range(8)),
        "clean_integer_range": [-127, 127],
        "faulted_integer_range": [-128, 127],
        "persistence": "same fault across all images of one scenario",
        "restoration": "verified shared clean state between scenarios",
    },
    "sampling": {
        "layers": "all_63_eligible_layers",
        "bits": "equal_allocation_across_8_bits",
        "weights": "uniform_without_replacement_within_layer_and_bit",
        "sign_or_zero_balancing": False,
        "sample_count": "PENDING_precision_and_compute_review",
        "hardware_fault_probability_assumed": False,
    },
    "evaluation": {
        "images": "frozen_1000_development_images",
        "ground_truth": "original_COCO_annotations",
        "method": "COCOeval_bbox_with_05a_inference_settings",
        "primary_layer_estimand": (
            "equal_bit_average_of_expected_nonnegative_AP50_95_loss"
        ),
        "nonnegative_loss": "max(0, clean_AP50_95 - faulty_AP50_95)",
        "additional_reports": [
            "signed_AP50_95_change_for_every_scenario",
            "per_bit_loss_distributions",
            "AP50_AP75_AR100",
            "observed_worst_case_not_population_worst_case",
            "nonfinite_outputs_and_execution_failures_separately",
        ],
    },
    "pending_before_freeze": [
        "Practical degradation thresholds and their rationale",
        "Fault sample counts, precision targets and compute budget",
        "Failure handling and inclusion rules",
        "Image-sampling and fault-sampling uncertainty",
        "Multiplicity and ranking-stability assessment",
        "Independent follow-up samples and selection rules",
        "Execution order, seeds and versioned stopping rules",
    ],
    "held_out_images_used": False,
    "limitations": [
        "Equal-bit weighting is a declared experimental distribution.",
        "Equal layer averages are not weight-population averages.",
        "Broad layer coverage does not establish sampling sufficiency.",
        "Fixed-image results do not establish dataset generalization.",
        "This is not a physical hardware reliability estimate.",
    ],
}

print(json.dumps(screening_draft, indent=2))
print("\nDraft only. No scenarios generated.")
print("No files modified. No model loaded. No inference performed.")

{
  "status": "DRAFT_NOT_FROZEN",
  "baseline": {
    "source": "05a_shared_INT8_clean_evaluation_v1.json",
    "representation": "logical_signed_INT8_weights_FP32_computation",
    "quantization": "per_output_channel_symmetric",
    "shared_clean_model_across_all_fault_scenarios": true
  },
  "scope": {
    "eligible_conv2d_layers": 63,
    "locations": "Conv2d weight elements only",
    "excluded": [
      "fixed DFL projection",
      "biases",
      "quantization scales",
      "activations",
      "other parameters and buffers"
    ],
    "claim": "logical weight-fault sensitivity within this scope"
  },
  "fault_model": {
    "operation": "single_bit_XOR_in_8bit_twos_complement",
    "bits": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7
    ],
    "clean_integer_range": [
      -127,
      127
    ],
    "faulted_integer_range": [
      -128,
      127
    ],
    "persistence": "same fault across all images of one scenario",
    "restoration": "verified

In [ ]:
from pathlib import Path
import gzip
import json
import statistics

directory = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/notebook05a/shared_clean_cpu_v1"
)

paths = sorted(directory.glob("chunk_*.json.gz"))
if len(paths) != 40:
    raise RuntimeError(f"Expected 40 chunks; found {len(paths)}.")

timing_fields = {}
total_bytes = 0

for index, path in enumerate(paths):
    total_bytes += path.stat().st_size
    envelope = json.loads(gzip.decompress(path.read_bytes()))
    payload = envelope["payload"]

    if index == 0:
        print("Chunk payload fields:", sorted(payload))

    for key, value in payload.items():
        if (
            any(word in key.lower() for word in ("seconds", "elapsed", "duration"))
            and isinstance(value, (int, float))
            and not isinstance(value, bool)
        ):
            timing_fields.setdefault(key, []).append(float(value))

print("\nPaired chunks:", len(paths))
print("Compressed predictions total (MiB):", round(total_bytes / 2**20, 3))

for key, values in timing_fields.items():
    print(f"\nTiming field: {key}")
    print("Recorded chunks:", len(values))
    print("Sum (seconds):", round(sum(values), 3))
    print("Median per chunk (seconds):", round(statistics.median(values), 3))
    print("Range (seconds):", [round(min(values), 3), round(max(values), 3)])

if not timing_fields:
    print("No top-level numeric timing fields found; inspect schema next.")

print("\nTimings describe the recorded paired CPU execution only.")
print("No inference performed. No files modified.")

Chunk payload fields: ['fingerprint', 'image_ids', 'per_image', 'predictions', 'status', 'wall_seconds']

Paired chunks: 40
Compressed predictions total (MiB): 8.438

Timing field: wall_seconds
Recorded chunks: 40
Sum (seconds): 1695.969
Median per chunk (seconds): 42.417
Range (seconds): [40.979, 45.52]

Timings describe the recorded paired CPU execution only.
No inference performed. No files modified.


In [ ]:
import math

layers = 63
bits = 8
images_per_scenario = 1000

observed_paired_cpu_seconds = 1695.969
cpu_seconds_per_forward = (
    observed_paired_cpu_seconds / (2 * images_per_scenario)
)

print("=== Illustrative compute planning ===")
print("Weights/bit/layer | Fault scenarios | Image forwards | CPU hours")

for n in (5, 10, 20, 40):
    scenarios = layers * bits * n
    forwards = scenarios * images_per_scenario
    hours = forwards * cpu_seconds_per_forward / 3600
    print(f"{n:17d} | {scenarios:15d} | {forwards:14d} | {hours:9.1f}")

print("\nCPU estimates use half the observed paired clean-run time.")
print("Faulted models may have different prediction density and runtime.")
print("Excludes evaluation, preparation, verification and interruption costs.")
print("These are neither GPU estimates nor guaranteed runtime bounds.")

print("\n=== Illustrative rare-event sampling ===")
print("Assumes independent draws with a fixed event probability.")
print("Event probability | n | Probability of observing at least one event")

for probability in (0.01, 0.05):
    for n in (5, 20, 40):
        detection_probability = 1 - (1 - probability) ** n
        print(f"{probability:17.1%} | {n:2d} | {detection_probability:10.2%}")

    required = math.ceil(
        math.log(0.05) / math.log(1 - probability)
    )
    print(
        f"Approximate draws for 95% chance of at least one event "
        f"at p={probability:.0%}: {required}"
    )

print("\nEvent probabilities are hypothetical, not measured fault rates.")
print("This is a discovery calculation, not precision or ranking validation.")
print("Actual sampling is without replacement; this is an approximation.")
print("No sample count selected. No files modified. No inference performed.")

=== Illustrative compute planning ===
Weights/bit/layer | Fault scenarios | Image forwards | CPU hours
                5 |            2520 |        2520000 |     593.6
               10 |            5040 |        5040000 |    1187.2
               20 |           10080 |       10080000 |    2374.4
               40 |           20160 |       20160000 |    4748.7

CPU estimates use half the observed paired clean-run time.
Faulted models may have different prediction density and runtime.
Excludes evaluation, preparation, verification and interruption costs.
These are neither GPU estimates nor guaranteed runtime bounds.

=== Illustrative rare-event sampling ===
Assumes independent draws with a fixed event probability.
Event probability | n | Probability of observing at least one event
             1.0% |  5 |      4.90%
             1.0% | 20 |     18.21%
             1.0% | 40 |     33.10%
Approximate draws for 95% chance of at least one event at p=1%: 299
             5.0% |  5 |     22.6

In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import torch

print("=== 05b: GPU Environment Check ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Drive accessible:", Path("/content/drive/MyDrive").is_dir())

for package in ["torchvision", "ultralytics", "numpy", "pycocotools"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; do not start the benchmark.")

print("GPU:", torch.cuda.get_device_name(0))
probe = torch.ones(1, device="cuda:0")
torch.cuda.synchronize()
print("CUDA tensor allocation: PASS")
del probe

print("No packages installed. No files modified. No model inference performed.")

=== 05b: GPU Environment Check ===
Python: 3.13.16
PyTorch: 2.14.1+cu130
PyTorch CUDA build: 13.0
CUDA available: True
Drive accessible: False
torchvision: 0.29.1+cu130
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
GPU: Tesla T4
CUDA tensor allocation: PASS
No packages installed. No files modified. No model inference performed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import subprocess
import sys
from importlib.metadata import version

packages = [
    "ultralytics==8.4.189",
    "nvidia-ml-py==13.615.71",
    "ultralytics-thop==2.2.5",
    "ultralytics-platform==0.1.108",
    "jedi==0.20.1",
]

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--disable-pip-version-check",
        *packages,
    ],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dependency installation failed.")

print("--- Installed versions ---")
for package in packages:
    name = package.split("==")[0]
    print(f"{name}: {version(name)}")

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)
print("\n--- Dependency check ---")
print(check.stdout.strip())
print(check.stderr.strip())
if check.returncode != 0:
    raise RuntimeError("Dependency compatibility check failed.")

import torch
import torchvision
import ultralytics

print("\nRequired imports: PASS")
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Ultralytics:", ultralytics.__version__)
print("No model loaded. No inference performed.")

--- Installed versions ---
ultralytics: 8.4.189
nvidia-ml-py: 13.615.71
ultralytics-thop: 2.2.5
ultralytics-platform: 0.1.108
jedi: 0.20.1

--- Dependency check ---
No broken requirements found.

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

Required imports: PASS
PyTorch: 2.14.1+cu130
torchvision: 0.29.1+cu130
Ultralytics: 8.4.189
No model loaded. No inference performed.


In [ ]:
from pathlib import Path
import hashlib
import json
import torch
from ultralytics import YOLO

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = root / "code"
checkpoint = root / "models/yolov8s.pt"
report_path = (
    repo / "docs/research_reports/"
    "05a_shared_INT8_clean_evaluation_v1.json"
)

def sha256(content):
    return hashlib.sha256(content).hexdigest()

if sha256(checkpoint.read_bytes()) != (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
):
    raise RuntimeError("Checkpoint SHA256 mismatch.")

report_bytes = report_path.read_bytes()
if sha256(report_bytes) != (
    "47861220401cc36ea2c6c19f427d351dcbb5ed39a9067fa79e0abbb175f889d7"
):
    raise RuntimeError("Published report SHA256 mismatch.")

report = json.loads(report_bytes)["evaluation_record"]["payload"]
config05a = report["config"]

torch.manual_seed(20261112)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

benchmark_model = YOLO(str(checkpoint)).model.cpu().float().eval()
quantized_layers = 0

with torch.no_grad():
    for name, module in benchmark_model.named_modules():
        if not isinstance(module, torch.nn.Conv2d):
            continue
        if name == "model.22.dfl.conv":
            continue

        maxima = module.weight.abs().amax(
            dim=(1, 2, 3), keepdim=True
        )
        scales = torch.where(
            maxima > 0, maxima / 127, torch.ones_like(maxima)
        )
        q = torch.round(
            module.weight / scales
        ).clamp(-127, 127).to(torch.int8)
        module.weight.copy_(q.float() * scales)
        quantized_layers += 1

if quantized_layers != 63:
    raise RuntimeError("Unexpected quantized layer count.")

with torch.inference_mode():
    benchmark_model(torch.zeros(1, 3, 640, 640))

def model_state_sha(model):
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        metadata = json.dumps(
            [name, str(array.dtype), list(array.shape)],
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
        digest.update(metadata)
        digest.update(array.tobytes())
    return digest.hexdigest()

expected_state = config05a["model_state_sha256"]["int8_channel_clean"]
if model_state_sha(benchmark_model) != expected_state:
    raise RuntimeError("Reconstructed INT8 state differs from 05a.")

benchmark_model = benchmark_model.to("cuda:0")
if model_state_sha(benchmark_model) != expected_state:
    raise RuntimeError("Model state changed during GPU transfer.")

print("Checkpoint and published report: PASS")
print("Quantized Conv2d layers:", quantized_layers)
print("INT8 model state matches 05a: PASS")
print("GPU transfer state integrity: PASS")
print("Device:", next(benchmark_model.parameters()).device)
print("One synthetic CPU forward completed for state preparation.")
print("No development images evaluated. No faults injected.")
print("No files modified. Benchmark not started.")

Checkpoint and published report: PASS
Quantized Conv2d layers: 63
INT8 model state matches 05a: PASS
GPU transfer state integrity: PASS
Device: cuda:0
One synthetic CPU forward completed for state preparation.
No development images evaluated. No faults injected.
No files modified. Benchmark not started.


In [ ]:
import json
import time

split_path = (
    repo / "experiments/notebook05/coco2017_split_manifest_v1.json"
)
split_bytes = split_path.read_bytes()

if sha256(split_bytes) != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest SHA256 mismatch.")

split = json.loads(split_bytes)
development = split["image_ids"]["development"]
held_out = split["image_ids"]["held_out"]
benchmark_ids = list(split["image_ids"]["pilot"])

if (
    len(benchmark_ids) != 50
    or len(set(benchmark_ids)) != 50
    or not set(benchmark_ids) <= set(development)
    or set(benchmark_ids) & set(held_out)
):
    raise RuntimeError("Invalid benchmark image selection.")

annotation_path = (
    root / "datasets/coco2017/annotations/instances_val2017.json"
)
annotation_bytes = annotation_path.read_bytes()
if sha256(annotation_bytes) != split["annotation_sha256"]:
    raise RuntimeError("Annotation SHA256 mismatch.")

annotation = json.loads(annotation_bytes)
benchmark_images = {
    image["id"]: image for image in annotation["images"]
}
benchmark_categories = [
    category["id"]
    for category in sorted(annotation["categories"], key=lambda row: row["id"])
]

benchmark_image_bytes = {}
benchmark_image_hashes = {}
started = time.perf_counter()

for image_id in benchmark_ids:
    filename = benchmark_images[image_id]["file_name"]
    if Path(filename).name != filename:
        raise RuntimeError("Unsafe image filename.")
    content = (
        root / "datasets/coco2017/images/val2017" / filename
    ).read_bytes()
    benchmark_image_bytes[image_id] = content
    benchmark_image_hashes[str(image_id)] = sha256(content)

benchmark_drive_read_seconds = time.perf_counter() - started

print("Frozen pilot images cached in memory:", len(benchmark_image_bytes))
print("Compressed image bytes (MiB):", round(
    sum(map(len, benchmark_image_bytes.values())) / 2**20, 3
))
print("Drive read and hashing seconds:", round(benchmark_drive_read_seconds, 3))
print("Pilot/development and held-out separation: PASS")
print("No inference performed. No files modified.")

Frozen pilot images cached in memory: 50
Compressed image bytes (MiB): 7.574
Drive read and hashing seconds: 43.89
Pilot/development and held-out separation: PASS
No inference performed. No files modified.


In [ ]:
import cv2
import numpy as np
import platform
import statistics
from importlib.metadata import version
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

benchmark_path = (
    repo / "results/notebook05b/"
    "shared_INT8_clean_gpu_timing_pilot50_v1.json"
)
if benchmark_path.exists():
    raise FileExistsError(
        "Timing report already exists. Review it before another benchmark."
    )

transform = LetterBox(
    new_shape=(640, 640),
    auto=False,
    scale_fill=False,
    scaleup=True,
    stride=32,
)

if model_state_sha(benchmark_model) != expected_state:
    raise RuntimeError("Model state differs from the verified baseline.")

with torch.inference_mode():
    warmup = torch.zeros(1, 3, 640, 640, device="cuda:0")
    for _ in range(5):
        benchmark_model(warmup)
    torch.cuda.synchronize()
    del warmup

timing_runs = []

with torch.inference_mode():
    for repeat in range(3):
        predictions = []
        degenerate_count = 0
        torch.cuda.synchronize()
        started = time.perf_counter()

        for image_id in benchmark_ids:
            image = cv2.imdecode(
                np.frombuffer(
                    benchmark_image_bytes[image_id], dtype=np.uint8
                ),
                cv2.IMREAD_COLOR,
            )
            record = benchmark_images[image_id]
            if image is None or image.shape[:2] != (
                record["height"], record["width"]
            ):
                raise RuntimeError(f"Invalid image: {image_id}")

            padded = transform(image=image)
            array = np.ascontiguousarray(
                padded[:, :, ::-1].transpose(2, 0, 1)
            )
            tensor = (
                torch.from_numpy(array).unsqueeze(0).float() / 255.0
            ).to("cuda:0")

            result = benchmark_model(tensor)
            raw = result[0] if isinstance(result, (tuple, list)) else result
            raw = raw.detach().cpu()

            if tuple(raw.shape) != (1, 84, 8400):
                raise RuntimeError("Unexpected raw output shape.")
            if not torch.isfinite(raw).all():
                raise RuntimeError("Non-finite raw predictions.")

            boxes = non_max_suppression(
                raw.clone(),
                conf_thres=0.001,
                iou_thres=0.7,
                nc=80,
                max_det=300,
            )[0].clone()
            boxes[:, :4] = scale_boxes(
                tensor.shape[2:], boxes[:, :4], image.shape[:2]
            )
            if not torch.isfinite(boxes).all():
                raise RuntimeError("Non-finite post-NMS predictions.")

            for box in boxes:
                x1, y1, x2, y2, score, cls = box.tolist()
                if x2 <= x1 or y2 <= y1:
                    degenerate_count += 1
                    continue
                if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                    raise RuntimeError("Invalid class or score.")
                predictions.append({
                    "image_id": image_id,
                    "category_id": benchmark_categories[int(cls)],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                })

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        prediction_sha = sha256(json.dumps(
            predictions, sort_keys=True, separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8"))

        timing_runs.append({
            "repeat": repeat + 1,
            "images": len(benchmark_ids),
            "wall_seconds": elapsed,
            "seconds_per_image": elapsed / len(benchmark_ids),
            "prediction_count": len(predictions),
            "prediction_sha256": prediction_sha,
            "degenerate_boxes_excluded": degenerate_count,
        })
        print(
            f"Repeat {repeat + 1}/3: {elapsed:.3f} seconds | "
            f"predictions={len(predictions)}"
        )

if model_state_sha(benchmark_model) != expected_state:
    raise RuntimeError("Model state changed during timing.")

timing_report = {
    "purpose": "clean_pipeline_compute_planning_only",
    "model_state_sha256": expected_state,
    "image_ids": benchmark_ids,
    "image_sha256": benchmark_image_hashes,
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": version("torchvision"),
        "ultralytics": version("ultralytics"),
        "opencv_python": version("opencv-python"),
        "gpu": torch.cuda.get_device_name(0),
        "torch_threads": torch.get_num_threads(),
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "matmul_tf32": torch.backends.cuda.matmul.allow_tf32,
        "cudnn_tf32": torch.backends.cudnn.allow_tf32,
    },
    "settings": config05a["settings"],
    "inference_device": "cuda:0",
    "nms_device": "cpu",
    "drive_read_and_hash_seconds": benchmark_drive_read_seconds,
    "timed_scope": (
        "decode_preprocess_transfer_forward_cpuNMS_coordinate_conversion"
    ),
    "excluded_costs": [
        "Drive reads", "model preparation", "warmup",
        "state hashing", "prediction hashing",
        "result saving", "COCOeval",
    ],
    "runs": timing_runs,
    "median_seconds_per_image": statistics.median(
        run["seconds_per_image"] for run in timing_runs
    ),
    "repeated_prediction_hashes_equal": (
        len({run["prediction_sha256"] for run in timing_runs}) == 1
    ),
    "limitations": [
        "Clean model timing only; no injected faults.",
        "Repeated fixed pilot images with cached compressed image bytes.",
        "Not a native INT8 hardware benchmark.",
        "Not a fault-campaign runtime guarantee.",
        "CPU/GPU detection accuracy agreement not established by this timing.",
    ],
}

content = (
    json.dumps(timing_report, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
benchmark_path.parent.mkdir(parents=True, exist_ok=True)
with benchmark_path.open("xb") as handle:
    handle.write(content)
if benchmark_path.read_bytes() != content:
    raise RuntimeError("Timing report write verification failed.")

print("Median seconds/image:", timing_report["median_seconds_per_image"])
print(
    "Repeated prediction hashes equal:",
    timing_report["repeated_prediction_hashes_equal"],
)
print("Model state unchanged: PASS")
print("Saved:", benchmark_path)
print("Report SHA256:", sha256(content))
print("No faults injected. No held-out images evaluated. No commit or push.")

Repeat 1/3: 1.286 seconds | predictions=4754
Repeat 2/3: 1.052 seconds | predictions=4754
Repeat 3/3: 1.083 seconds | predictions=4754
Median seconds/image: 0.02166507941999953
Repeated prediction hashes equal: True
Model state unchanged: PASS
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05b/shared_INT8_clean_gpu_timing_pilot50_v1.json
Report SHA256: 3cbd9a01c0e525bd2c4601b119aec7039377c5ceb18d7eb60eb1932e6480af02
No faults injected. No held-out images evaluated. No commit or push.


In [ ]:
from pathlib import Path
import hashlib
import json

timing_path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/notebook05b/shared_INT8_clean_gpu_timing_pilot50_v1.json"
)
content = timing_path.read_bytes()

if hashlib.sha256(content).hexdigest() != (
    "3cbd9a01c0e525bd2c4601b119aec7039377c5ceb18d7eb60eb1932e6480af02"
):
    raise RuntimeError("Timing report SHA256 mismatch.")

timing = json.loads(content)
seconds_per_image = timing["median_seconds_per_image"]

print("Illustrative screening compute — not a frozen design")
print("Weights/bit | Scenarios | Images/scenario | Image forwards | Hours")

for weights_per_bit in (5, 10, 20):
    scenarios = 63 * 8 * weights_per_bit
    for image_count in (50, 200, 1000):
        forwards = scenarios * image_count
        hours = forwards * seconds_per_image / 3600
        print(
            f"{weights_per_bit:11d} | {scenarios:9d} | "
            f"{image_count:15d} | {forwards:14d} | {hours:5.2f}"
        )

print("\nEstimates cover the measured cached-image clean pipeline only.")
print("Fault effects may change prediction density and processing time.")
print("Excludes preparation, integrity checks, saving and COCOeval.")
print("No statistical sufficiency follows from these compute estimates.")
print("No design selected. No inference performed. No files modified.")

Illustrative screening compute — not a frozen design
Weights/bit | Scenarios | Images/scenario | Image forwards | Hours
          5 |      2520 |              50 |         126000 |  0.76
          5 |      2520 |             200 |         504000 |  3.03
          5 |      2520 |            1000 |        2520000 | 15.17
         10 |      5040 |              50 |         252000 |  1.52
         10 |      5040 |             200 |        1008000 |  6.07
         10 |      5040 |            1000 |        5040000 | 30.33
         20 |     10080 |              50 |         504000 |  3.03
         20 |     10080 |             200 |        2016000 | 12.13
         20 |     10080 |            1000 |       10080000 | 60.66

Estimates cover the measured cached-image clean pipeline only.
Fault effects may change prediction density and processing time.
Excludes preparation, integrity checks, saving and COCOeval.
No statistical sufficiency follows from these compute estimates.
No design selected. No

In [ ]:
from pathlib import Path
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
timing_path = (
    repo / "results/notebook05b/"
    "shared_INT8_clean_gpu_timing_pilot50_v1.json"
)
timing_bytes = timing_path.read_bytes()
timing_sha = hashlib.sha256(timing_bytes).hexdigest()

if timing_sha != (
    "3cbd9a01c0e525bd2c4601b119aec7039377c5ceb18d7eb60eb1932e6480af02"
):
    raise RuntimeError("Timing report SHA256 mismatch.")

timing = json.loads(timing_bytes)
scenarios = 63 * 8 * 10
image_forwards = scenarios * 200

planning = {
    "status": "WORKING_CANDIDATE_NOT_FROZEN",
    "baseline_commit": "30661e980b0854e92cb36fde105f5723cb14ef39",
    "baseline": "shared_logical_INT8_per_channel_FP32_computation",
    "timing_source": timing_path.relative_to(repo).as_posix(),
    "timing_source_sha256": timing_sha,
    "observed_clean_median_seconds_per_image": (
        timing["median_seconds_per_image"]
    ),
    "working_candidate": {
        "eligible_layers": 63,
        "bits": list(range(8)),
        "uniform_weights_per_layer_bit": 10,
        "fault_scenarios": scenarios,
        "development_images_per_scenario": 200,
        "image_forwards": image_forwards,
        "image_subset": "NOT_YET_SELECTED",
        "reason": "budget-based candidate; not a power-validated sample size",
    },
    "estimated_measured_pipeline_hours": (
        image_forwards * timing["median_seconds_per_image"] / 3600
    ),
    "estimate_excludes": [
        "fault injection and restoration",
        "integrity checks",
        "prediction storage",
        "COCOeval",
        "interruptions and resume overhead",
    ],
    "required_before_freeze": [
        "Time and validate a complete faulted scenario.",
        "Specify the fixed development image subset and audit coverage.",
        "Define practical damage thresholds and uncertainty analysis.",
        "Define independent follow-up faults and image expansion.",
        "Include confirmation controls from apparently low-risk layers.",
        "Specify failure handling, selection rules and stopping rules.",
        "Freeze seeds, execution order and source identities.",
    ],
    "limitations": [
        "Clean timing does not guarantee faulted-scenario runtime.",
        "Ten weights per bit have no established precision guarantee.",
        "Screening does not guarantee discovery of rare damaging faults.",
        "No physical hardware fault-rate claim.",
    ],
    "new_fault_scenarios_generated": False,
    "held_out_images_used": False,
}

destination = (
    repo / "experiments/notebook05b/"
    "screening_compute_planning_v1.json"
)
content = (
    json.dumps(planning, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing planning record differs; do not overwrite.")
    print("Existing planning record matches: PASS")
else:
    with destination.open("xb") as handle:
        handle.write(content)

if destination.read_bytes() != content:
    raise RuntimeError("Planning record verification failed.")

print("Saved:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Status:", planning["status"])
print("No fault campaign started. No commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05b/screening_compute_planning_v1.json
SHA256: 809ba121fd0e1f9999d89d901e3027983b29e47425dbb9a1b8f7f0c08df02cdc
Status: WORKING_CANDIDATE_NOT_FROZEN
No fault campaign started. No commit or push performed.


In [ ]:
import hashlib
import json
from collections import Counter
import numpy as np

split_path = (
    repo / "experiments/notebook05/coco2017_split_manifest_v1.json"
)
split_bytes = split_path.read_bytes()
if hashlib.sha256(split_bytes).hexdigest() != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest SHA256 mismatch.")

split = json.loads(split_bytes)
development_ids = sorted(split["image_ids"]["development"])
held_out_ids = set(split["image_ids"]["held_out"])

annotation_bytes = (
    root / "datasets/coco2017/annotations/instances_val2017.json"
).read_bytes()
if hashlib.sha256(annotation_bytes).hexdigest() != split["annotation_sha256"]:
    raise RuntimeError("Annotation SHA256 mismatch.")
coco = json.loads(annotation_bytes)

screening_image_seed = 20261113
rng = np.random.Generator(np.random.PCG64(screening_image_seed))
screening_candidate_ids = sorted(
    int(value)
    for value in rng.permutation(development_ids)[:200]
)
candidate_set = set(screening_candidate_ids)

if (
    len(candidate_set) != 200
    or not candidate_set <= set(development_ids)
    or candidate_set & held_out_ids
):
    raise RuntimeError("Invalid screening image subset.")

category_names = {
    category["id"]: category["name"]
    for category in coco["categories"]
}
box_counts = Counter()
category_image_ids = {
    category_id: set() for category_id in category_names
}
images_with_valid_boxes = set()

for annotation in coco["annotations"]:
    if annotation["image_id"] not in candidate_set:
        continue
    if annotation.get("iscrowd", 0):
        continue
    _, _, width, height = annotation["bbox"]
    if width <= 0 or height <= 0:
        continue

    category_id = annotation["category_id"]
    box_counts[category_id] += 1
    category_image_ids[category_id].add(annotation["image_id"])
    images_with_valid_boxes.add(annotation["image_id"])

ids_bytes = json.dumps(
    screening_candidate_ids, separators=(",", ":")
).encode("utf-8")

print("Status: CANDIDATE_NOT_FROZEN")
print("Sampling: PCG64 permutation of sorted development image IDs")
print("Seed:", screening_image_seed)
print("Images:", len(screening_candidate_ids))
print("Image ID list SHA256:", hashlib.sha256(ids_bytes).hexdigest())
print("Categories with valid non-crowd boxes:", len(box_counts))
print("Missing categories:", [
    category_names[category_id]
    for category_id in sorted(category_names)
    if not box_counts[category_id]
])
print("Valid non-crowd boxes:", sum(box_counts.values()))
print("Images without valid non-crowd boxes:", (
    len(candidate_set - images_with_valid_boxes)
))
print("Overlap with the historical pilot:", len(
    candidate_set & set(split["image_ids"]["pilot"])
))

print("\nLeast represented categories present:")
for category_id in sorted(
    box_counts,
    key=lambda key: (len(category_image_ids[key]), box_counts[key], key),
)[:10]:
    print(
        category_names[category_id],
        "| images:", len(category_image_ids[category_id]),
        "| boxes:", box_counts[category_id],
    )

print("\nCoverage audit only; no statistical sufficiency claim.")
print("No files modified. No inference performed.")

Status: CANDIDATE_NOT_FROZEN
Sampling: PCG64 permutation of sorted development image IDs
Seed: 20261113
Images: 200
Image ID list SHA256: 88c187b86b039a9b3ff69909422c0fd2a30108aa550ce19667d6bb06c98a8293
Categories with valid non-crowd boxes: 76
Missing categories: ['parking meter', 'carrot', 'hot dog', 'hair drier']
Valid non-crowd boxes: 1311
Images without valid non-crowd boxes: 4
Overlap with the historical pilot: 13

Least represented categories present:
fire hydrant | images: 1 | boxes: 1
kite | images: 1 | boxes: 1
toaster | images: 1 | boxes: 1
toothbrush | images: 1 | boxes: 1
donut | images: 1 | boxes: 5
umbrella | images: 2 | boxes: 2
snowboard | images: 2 | boxes: 2
mouse | images: 2 | boxes: 2
keyboard | images: 2 | boxes: 2
scissors | images: 2 | boxes: 2

Coverage audit only; no statistical sufficiency claim.
No files modified. No inference performed.


In [ ]:
from pathlib import Path
import hashlib
import json

expected_ids_sha = (
    "88c187b86b039a9b3ff69909422c0fd2a30108aa550ce19667d6bb06c98a8293"
)
ids_bytes = json.dumps(
    screening_candidate_ids, separators=(",", ":")
).encode("utf-8")

if hashlib.sha256(ids_bytes).hexdigest() != expected_ids_sha:
    raise RuntimeError("Candidate image IDs differ from the audited subset.")

manifest = {
    "status": "IMAGE_SUBSET_FROZEN_CAMPAIGN_NOT_FINALIZED",
    "purpose": "initial_screening_and_compute_validation",
    "selection": {
        "algorithm": "PCG64_permutation_of_sorted_development_image_ids",
        "seed": screening_image_seed,
        "numpy_version": np.__version__,
        "image_count": len(screening_candidate_ids),
        "image_ids": screening_candidate_ids,
        "image_id_list_sha256": expected_ids_sha,
        "source_split_sha256": hashlib.sha256(split_bytes).hexdigest(),
        "annotation_sha256": split["annotation_sha256"],
    },
    "coverage": {
        "categories_present": len(box_counts),
        "missing_categories": [
            category_names[c]
            for c in sorted(category_names)
            if not box_counts[c]
        ],
        "valid_non_crowd_boxes": sum(box_counts.values()),
        "images_without_valid_non_crowd_boxes": sorted(
            candidate_set - images_with_valid_boxes
        ),
        "per_category": [
            {
                "category_id": c,
                "name": category_names[c],
                "images": len(category_image_ids[c]),
                "valid_non_crowd_boxes": box_counts[c],
            }
            for c in sorted(category_names)
        ],
    },
    "rules": [
        "Retain all 200 images, including images without valid non-crowd boxes.",
        "Use original COCO annotations and COCOeval crowd handling.",
        "Compare clean and faulted predictions on identical image IDs.",
        "Do not change the subset in response to performance results.",
        "Do not use held-out images for screening decisions.",
    ],
    "limitations": [
        "Four categories have no valid non-crowd boxes in this subset.",
        "Rare-category sensitivity cannot be established by this screening.",
        "No sample-size or ranking-stability guarantee.",
    ],
}

destination = (
    repo / "experiments/notebook05b/screening_image_manifest_v1.json"
)
content = (
    json.dumps(manifest, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

destination.parent.mkdir(parents=True, exist_ok=True)
if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing manifest differs; do not overwrite.")
else:
    with destination.open("xb") as handle:
        handle.write(content)

if destination.read_bytes() != content:
    raise RuntimeError("Manifest verification failed.")

print("Saved:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Frozen image count:", len(screening_candidate_ids))
print("Campaign design remains pending.")
print("No inference, commit or push performed.")


Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05b/screening_image_manifest_v1.json
SHA256: f2130cb0b0d505d641c8604dcb075cdcd0830092e5d8b31f594a45dcf1074aeb
Frozen image count: 200
Campaign design remains pending.
No inference, commit or push performed.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import io
import subprocess
import sys
import zipfile

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one ZIP file.")

content = next(iter(uploaded.values()))
if hashlib.sha256(content).hexdigest() != (
    "5ffb655376e35d44615760f51d2de50b8c045b37e1cbd0ba5ee9cb9c9db57bb2"
):
    raise RuntimeError("Package SHA256 mismatch.")

expected = {
    "src/fault05b_validation_v1.py":
        "c84cdb64cc5774a0645dce44f904aed516169f3729102cf1a3230b66e463d8c6",
    "tests/test_fault05b_validation_v1.py":
        "e445316b8f4b686b66e5b47c8dba659f7f84127777da7eef5e3d911c71b25741",
    "docs/research_reports/05b_validation_runner_notes_v1.md":
        "3210bbce95acb0c33d96aab14cc370a70f4fa102ae2fa901c07ccc805bd83a2b",
}

with zipfile.ZipFile(io.BytesIO(content)) as archive:
    names = archive.namelist()
    if len(names) != len(expected) or set(names) != set(expected):
        raise RuntimeError("Unexpected package entries.")

    verified = {}
    for name, expected_sha in expected.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected_sha:
            raise RuntimeError(f"File checksum mismatch: {name}")
        destination = repo / name
        if destination.exists() and destination.read_bytes() != data:
            raise RuntimeError(f"Existing file differs: {name}")
        verified[name] = data

    for name, data in verified.items():
        destination = repo / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            with destination.open("xb") as handle:
                handle.write(data)
        if destination.read_bytes() != data:
            raise RuntimeError(f"Installation verification failed: {name}")
        print("Verified:", name)

result = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", "tests", "-p", "test_fault05b_validation_v1.py", "-v",
    ],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Validation module tests failed.")

print("Package installed. Synthetic tests passed.")
print("CUDA inference and restoration have not been tested by this cell.")
print("No model loaded. No inference, commit or push performed.")

Saving fault05b_validation_v1.zip to fault05b_validation_v1.zip
Verified: src/fault05b_validation_v1.py
Verified: tests/test_fault05b_validation_v1.py
Verified: docs/research_reports/05b_validation_runner_notes_v1.md

test_all_bit_flips_are_involutions_and_change_value (test_fault05b_validation_v1.BitAndScopeTests.test_all_bit_flips_are_involutions_and_change_value) ... ok
test_all_signed_values_and_bits_against_binary_encoding (test_fault05b_validation_v1.BitAndScopeTests.test_all_signed_values_and_bits_against_binary_encoding) ... ok
test_duplicate_wrong_size_and_overlap_rejected (test_fault05b_validation_v1.BitAndScopeTests.test_duplicate_wrong_size_and_overlap_rejected) ... ok
test_invalid_integer_and_bit_rejected (test_fault05b_validation_v1.BitAndScopeTests.test_invalid_integer_and_bit_rejected) ... ok
test_sign_bit_boundaries (test_fault05b_validation_v1.BitAndScopeTests.test_sign_bit_boundaries) ... ok
test_source_checksum_tampering_rejected (test_fault05b_validation_v1.BitAndS

In [ ]:
from pathlib import Path
import hashlib
import importlib.util

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
module_path = root / "code/src/fault05b_validation_v1.py"

expected_sha = (
    "c84cdb64cc5774a0645dce44f904aed516169f3729102cf1a3230b66e463d8c6"
)
if hashlib.sha256(module_path.read_bytes()).hexdigest() != expected_sha:
    raise RuntimeError("Validation module SHA256 mismatch.")

spec = importlib.util.spec_from_file_location(
    "fault05b_validation_v1", module_path
)
validation_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validation_module)

validation_result = validation_module.run(root)

print("\n=== Engineering validation summary ===")
print("Scenario:", validation_result["scenario"])
print("Clean:", validation_result["metrics"]["clean"])
print("Faulty:", validation_result["metrics"]["faulty"])
print(
    "Faulty minus clean AP50:95:",
    validation_result["faulty_minus_clean"]["AP50_95"],
)
print(
    "Nonnegative AP50:95 loss:",
    validation_result["nonnegative_AP50_95_loss"],
)
print("Timing seconds:", validation_result["timing_seconds"])
print(
    "Exact restoration verified:",
    validation_result["exact_restoration_verified"],
)
print("This convenience scenario is not layer-ranking evidence.")

INT8 clean: 200 images | 19809 predictions | 4.825 pipeline seconds
INT8 sign-bit engineering fault: 200 images | 19728 predictions | 4.620 pipeline seconds
Exact shared clean restoration: PASS
loading annotations into memory...
Done (t=0.91s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.04s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.73s).
Accumulating evaluation results...
DONE (t=0.64s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.484
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.624
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.533
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.275
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.482
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.706
 Average Recall     (AR) @[ IoU

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import io
import subprocess
import sys
import zipfile

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one ZIP file.")

content = next(iter(uploaded.values()))
if hashlib.sha256(content).hexdigest() != (
    "e3faceb158cae4f7ec93aae2d9ebe37ac21f623b2282c528a1c27e2ed58e6024"
):
    raise RuntimeError("Package SHA256 mismatch.")

expected = {
    "src/screen05b_campaign_v1.py":
        "f205c78e7bb0c026a25e8f3c097dcc06cbb927794e793ec66a333179563c40e8",
    "tests/test_screen05b_campaign_v1.py":
        "014928a394febe8b4c5eb95b00e9c6a02cf4eec13f0b5716a26d2df7946504ff",
    "docs/research_reports/05b_campaign_runner_notes_v1.md":
        "ad077a1eef21dcd579d1500feb671e112a4de4e3e2864cae9a0424d63064c6fa",
}

with zipfile.ZipFile(io.BytesIO(content)) as archive:
    names = archive.namelist()
    if len(names) != len(expected) or set(names) != set(expected):
        raise RuntimeError("Unexpected package entries.")

    verified = {}
    for name, expected_sha in expected.items():
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected_sha:
            raise RuntimeError(f"File checksum mismatch: {name}")
        destination = repo / name
        if destination.exists() and destination.read_bytes() != data:
            raise RuntimeError(f"Existing file differs: {name}")
        verified[name] = data

    for name, data in verified.items():
        destination = repo / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            with destination.open("xb") as handle:
                handle.write(data)
        if destination.read_bytes() != data:
            raise RuntimeError(f"Installation verification failed: {name}")
        print("Verified:", name)

result = subprocess.run(
    [
        sys.executable, "-m", "unittest", "discover",
        "-s", "tests", "-p", "test_screen05b_campaign_v1.py", "-v",
    ],
    cwd=repo,
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError("Campaign control tests failed.")

print("Campaign package installed. Control tests passed.")
print("No screening plan created. No model inference performed.")
print("CUDA execution remains to be validated.")
print("Existing research files unchanged. No commit or push.")

Saving screen05b_campaign_v1.zip to screen05b_campaign_v1.zip
Verified: src/screen05b_campaign_v1.py
Verified: tests/test_screen05b_campaign_v1.py
Verified: docs/research_reports/05b_campaign_runner_notes_v1.md

test_equal_bit_mean_does_not_cancel_harm (test_screen05b_campaign_v1.CampaignControlTests.test_equal_bit_mean_does_not_cancel_harm) ... ok
test_exclusive_write_and_partial_preservation (test_screen05b_campaign_v1.CampaignControlTests.test_exclusive_write_and_partial_preservation) ... ok
test_incomplete_or_unbalanced_summaries_rejected (test_screen05b_campaign_v1.CampaignControlTests.test_incomplete_or_unbalanced_summaries_rejected) ... ok
test_invalid_bit_values (test_screen05b_campaign_v1.CampaignControlTests.test_invalid_bit_values) ... ok
test_no_ranking_on_partial_campaign (test_screen05b_campaign_v1.CampaignControlTests.test_no_ranking_on_partial_campaign) ... ok
test_nonfinite_or_negative_losses_rejected (test_screen05b_campaign_v1.CampaignControlTests.test_nonfinite_or_n

In [ ]:
from pathlib import Path
import hashlib
import importlib.util

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
module_path = root / "code/src/screen05b_campaign_v1.py"

expected_sha = (
    "f205c78e7bb0c026a25e8f3c097dcc06cbb927794e793ec66a333179563c40e8"
)
if hashlib.sha256(module_path.read_bytes()).hexdigest() != expected_sha:
    raise RuntimeError("Campaign module SHA256 mismatch.")

spec = importlib.util.spec_from_file_location(
    "screen05b_campaign_v1", module_path
)
campaign_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(campaign_module)

campaign = campaign_module.Campaign(root)
screening_plan = campaign.create_plan()

print("\n=== Frozen exploratory screening plan ===")
print("Status:", screening_plan["status"])
print("Layers:", len(screening_plan["inventory"]))
print("Images/scenario:", len(screening_plan["image_ids"]))
print("Scenarios:", screening_plan["scenario_count"])
print("First two scenarios:")
for scenario in screening_plan["scenarios"][:2]:
    print(scenario)

print("\nUncertainty:", screening_plan["uncertainty"])
print("Confirmation:", screening_plan["confirmation"])
print("No development-image inference. No commit or push.")

Saved plan: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05b/int8_screening_plan_v1.json
Plan SHA256: 9981145a30329b326534faaae8fdffc7be911d1169b3869c081de612838caa6b
Scenarios: 5040 | no screening inference performed

=== Frozen exploratory screening plan ===
Status: FIXED_BUDGET_EXPLORATORY_SCREENING
Layers: 63
Images/scenario: 200
Scenarios: 5040
First two scenarios:
{'layer': 'model.22.cv2.2.1.conv', 'bit': 6, 'replicate': 8, 'flat_index': 2853, 'scenario_id': 'S05B-00001', 'q_clean': 2, 'q_faulty': 66, 'scale': 0.0008862112299539149, 'clean_weight': 0.0017724224599078298, 'faulty_weight': 0.05848994106054306}
{'layer': 'model.18.cv2.conv', 'bit': 5, 'replicate': 8, 'flat_index': 77798, 'scenario_id': 'S05B-00002', 'q_clean': -8, 'q_faulty': -40, 'scale': 0.0005805548862554133, 'clean_weight': -0.004644439090043306, 'faulty_weight': -0.023222194984555244}

Uncertainty: {'fault_bootstrap': '2000_stratified_percentile_draws_per_layer', 'seed': 20261115, 'con

In [ ]:
from pathlib import Path
import hashlib

plan_path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "experiments/notebook05b/int8_screening_plan_v1.json"
)

if hashlib.sha256(plan_path.read_bytes()).hexdigest() != (
    "9981145a30329b326534faaae8fdffc7be911d1169b3869c081de612838caa6b"
):
    raise RuntimeError("Frozen screening plan SHA256 mismatch.")

screening_gate_result = campaign.run(
    max_new_scenarios=2,
    max_wall_seconds=900,
)

print("\n=== Two-scenario execution check ===")
print("Status:", screening_gate_result["status"])
print("Completed:", screening_gate_result["completed_scenarios"])
print("Planned:", screening_gate_result["planned_scenarios"])
print("Clean metrics:", screening_gate_result["clean_metrics"])
print("No layer ranking is inferred from this implementation check.")

1/5040 | S05B-00001 | AP change=+0.00003993 | saved
2/5040 | S05B-00002 | AP change=-0.00001329 | saved
Completed: 2 /5040 | new: 2
Invocation wall seconds: 32.29
Summary: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook05b_screening_v1/progress_00002.json
No held-out evaluation. No protection or final ranking claim.

=== Two-scenario execution check ===
Status: PARTIAL_SCREENING_NO_RANKING
Completed: 2
Planned: 5040
Clean metrics: {'AP50_95': 0.4842769592793853, 'AP50': 0.6235200687849605, 'AP75': 0.5334446793894683, 'AP_small': 0.2747746377831853, 'AP_medium': 0.4817085008041767, 'AP_large': 0.7057658898369708, 'AR1': 0.40087117736099587, 'AR10': 0.5929026768900507, 'AR100': 0.6255526562312993, 'AR_small': 0.39721994235540353, 'AR_medium': 0.6273432880883161, 'AR_large': 0.8256609326346168}
No layer ranking is inferred from this implementation check.


In [ ]:
from pathlib import Path
import hashlib
import shutil

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
module_path = repo / "src/screen05b_campaign_v1.py"
plan_path = repo / "experiments/notebook05b/int8_screening_plan_v1.json"
output_dir = repo / "results/notebook05b/int8_screening_v1"

for path, expected_sha in [
    (
        module_path,
        "f205c78e7bb0c026a25e8f3c097dcc06cbb927794e793ec66a333179563c40e8",
    ),
    (
        plan_path,
        "9981145a30329b326534faaae8fdffc7be911d1169b3869c081de612838caa6b",
    ),
]:
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
        raise RuntimeError(f"Source identity mismatch: {path.name}")

prediction_paths = [
    output_dir / f"S05B-{index:05d}_predictions.json.gz"
    for index in (1, 2)
]
observed_max_bytes = max(path.stat().st_size for path in prediction_paths)
remaining = 5040 - len(list(output_dir.glob("S05B-*_evaluation.json.gz")))

estimated_prediction_bytes = observed_max_bytes * remaining
planning_reserve_bytes = int(estimated_prediction_bytes * 1.5) + 2**30
reported_free_bytes = shutil.disk_usage(output_dir).free

print(
    "Estimated remaining prediction storage (GiB):",
    round(estimated_prediction_bytes / 2**30, 2),
)
print(
    "Planning allowance with margin (GiB):",
    round(planning_reserve_bytes / 2**30, 2),
)
print(
    "Filesystem-reported free space (GiB):",
    round(reported_free_bytes / 2**30, 2),
)
print("Storage estimates are approximate; mounted Drive quota may differ.")

if reported_free_bytes < planning_reserve_bytes:
    raise RuntimeError(
        "Reported free space is below the planning allowance. "
        "Campaign was not started."
    )

screening_result = campaign.run(
    max_new_scenarios=None,
    max_wall_seconds=4 * 60 * 60,
)

completed = screening_result["completed_scenarios"]
planned = screening_result["planned_scenarios"]

print("\n=== Automatic execution checkpoint ===")
print("Status:", screening_result["status"])
print(f"Completed: {completed}/{planned} ({100 * completed / planned:.2f}%)")
print("Completed scenario artifacts are saved.")
print("No commit or push performed.")

Estimated remaining prediction storage (GiB): 3.74
Planning allowance with margin (GiB): 6.61
Filesystem-reported free space (GiB): 8.48
Storage estimates are approximate; mounted Drive quota may differ.
3/5040 | S05B-00003 | AP change=+0.00001219 | saved
4/5040 | S05B-00004 | AP change=+0.00000002 | saved
5/5040 | S05B-00005 | AP change=-0.00105221 | saved
6/5040 | S05B-00006 | AP change=-0.00196823 | saved
7/5040 | S05B-00007 | AP change=-0.00079016 | saved
8/5040 | S05B-00008 | AP change=+0.00002819 | saved
9/5040 | S05B-00009 | AP change=+0.00005721 | saved
10/5040 | S05B-00010 | AP change=-0.00133783 | saved
11/5040 | S05B-00011 | AP change=-0.00006655 | saved
12/5040 | S05B-00012 | AP change=-0.00000024 | saved
13/5040 | S05B-00013 | AP change=-0.00018509 | saved
14/5040 | S05B-00014 | AP change=-0.00000071 | saved
15/5040 | S05B-00015 | AP change=-0.00022968 | saved
16/5040 | S05B-00016 | AP change=+0.00001093 | saved
17/5040 | S05B-00017 | AP change=-0.00090302 | saved
18/5040 

In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("=== Screening Resume: Environment Check ===")
print("Python:", sys.version.split()[0])
print("Drive accessible:", Path("/content/drive/MyDrive").is_dir())

for package in [
    "torch", "torchvision", "ultralytics",
    "numpy", "pycocotools", "opencv-python", "Pillow",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("PyTorch CUDA build:", torch.version.cuda)
    print("PyTorch threads:", torch.get_num_threads())
except ImportError:
    print("PyTorch unavailable.")

print("No packages installed. No files modified. No inference performed.")

=== Screening Resume: Environment Check ===
Python: 3.13.18
Drive accessible: False
torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
opencv-python: 5.0.0.93
Pillow: 11.3.0
CUDA available: False
PyTorch threads: 1
No packages installed. No files modified. No inference performed.


In [5]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from pathlib import Path
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
output_dir = repo / "results/notebook05b/int8_screening_v1"
progress_path = (
    repo / "docs/research_reports/notebook05b_screening_v1/"
    "progress_01579.json"
)
plan_path = repo / "experiments/notebook05b/int8_screening_plan_v1.json"

plan_bytes = plan_path.read_bytes()
plan_sha = hashlib.sha256(plan_bytes).hexdigest()
if plan_sha != (
    "9981145a30329b326534faaae8fdffc7be911d1169b3869c081de612838caa6b"
):
    raise RuntimeError("Frozen plan SHA256 mismatch.")

plan = json.loads(plan_bytes)
progress_bytes = progress_path.read_bytes()
progress = json.loads(progress_bytes)

if progress["plan_sha256"] != plan_sha:
    raise RuntimeError("Progress report refers to a different plan.")

completed = progress["completed_scenarios"]
expected_ids = {
    row["scenario_id"] for row in plan["scenarios"][:completed]
}
evaluation_ids = {
    path.name.removesuffix("_evaluation.json.gz")
    for path in output_dir.glob("S05B-*_evaluation.json.gz")
}
prediction_ids = {
    path.name.removesuffix("_predictions.json.gz")
    for path in output_dir.glob("S05B-*_predictions.json.gz")
}

missing_evaluations = sorted(expected_ids - evaluation_ids)
missing_predictions = sorted(expected_ids - prediction_ids)
partial_files = sorted(output_dir.rglob("*.part"))
failure_files = sorted(output_dir.glob("*_failure.json.gz"))

print("Progress status:", progress["status"])
print(f"Recorded progress: {completed}/{plan['scenario_count']}")
print("Evaluation files:", len(evaluation_ids))
print("Prediction files:", len(prediction_ids))
print("Missing recorded evaluations:", missing_evaluations)
print("Missing recorded predictions:", missing_predictions)
print("Predictions without evaluations:", sorted(
    prediction_ids - evaluation_ids
))
print("Partial files:", [str(path.relative_to(output_dir)) for path in partial_files])
print("Failure reports:", [path.name for path in failure_files])
print(
    "Progress report SHA256:",
    hashlib.sha256(progress_bytes).hexdigest(),
)

if missing_evaluations or missing_predictions or partial_files or failure_files:
    raise RuntimeError("Checkpoint requires review before continuation.")

print("Recorded checkpoint file presence: PASS")
print("Prediction contents and checksums have not been audited by this cell.")
print("No inference, file changes, commit or push performed.")

Progress status: PARTIAL_SCREENING_NO_RANKING
Recorded progress: 1579/5040
Evaluation files: 1579
Prediction files: 1579
Missing recorded evaluations: []
Missing recorded predictions: []
Predictions without evaluations: []
Partial files: []
Failure reports: []
Progress report SHA256: f131052e115ca584fddda89a445349e014c7513b5f87e30b7bb07865904ce801
Recorded checkpoint file presence: PASS
Prediction contents and checksums have not been audited by this cell.
No inference, file changes, commit or push performed.


In [7]:
from pathlib import Path
import subprocess

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def git(*args):
    result = subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()

for title, args in [
    ("Branch", ["branch", "--show-current"]),
    ("HEAD", ["rev-parse", "HEAD"]),
    ("Tracked changes", ["diff", "--stat"]),
    ("Staged changes", ["diff", "--cached", "--stat"]),
]:
    print(f"\n--- {title} ---")
    print(git(*args) or "(none)")

untracked = git(
    "ls-files", "--others", "--exclude-standard"
).splitlines()

print("\n--- Untracked publication candidates ---")
for relative_path in untracked:
    path = repo / relative_path
    print(f"{relative_path} | bytes={path.stat().st_size}")

print("\nUntracked files:", len(untracked))
print(
    "Total untracked file bytes:",
    sum((repo / name).stat().st_size for name in untracked),
)
print("\nNo files changed or staged. No commit or push performed.")


--- Branch ---
main

--- HEAD ---
30661e980b0854e92cb36fde105f5723cb14ef39

--- Tracked changes ---
(none)

--- Staged changes ---
(none)

--- Untracked publication candidates ---
docs/research_reports/05b_campaign_runner_notes_v1.md | bytes=6715
docs/research_reports/05b_validation_runner_notes_v1.md | bytes=2336
docs/research_reports/notebook05b_screening_v1/progress_00002.json | bytes=8914
docs/research_reports/notebook05b_screening_v1/progress_00050.json | bytes=8915
docs/research_reports/notebook05b_screening_v1/progress_00100.json | bytes=8916
docs/research_reports/notebook05b_screening_v1/progress_00150.json | bytes=8916
docs/research_reports/notebook05b_screening_v1/progress_00200.json | bytes=8916
docs/research_reports/notebook05b_screening_v1/progress_00250.json | bytes=8916
docs/research_reports/notebook05b_screening_v1/progress_00300.json | bytes=8918
docs/research_reports/notebook05b_screening_v1/progress_00350.json | bytes=8920
docs/research_reports/notebook05b_screening

In [8]:
from pathlib import Path
import csv
import gzip
import hashlib
import io
import json
import math

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
output_dir = repo / "results/notebook05b/int8_screening_v1"
destination = (
    repo / "docs/research_reports/notebook05b_checkpoint_01579_v1"
)

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":"),
    ).encode("utf-8")

def sha256(content):
    return hashlib.sha256(content).hexdigest()

def read_record(path):
    content = path.read_bytes()
    envelope = json.loads(gzip.decompress(content))
    payload = envelope["payload"]
    if sha256(canonical(payload)) != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload checksum mismatch: {path.name}")
    return payload, sha256(content)

plan_path = repo / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()
plan_sha = sha256(plan_bytes)
if plan_sha != (
    "9981145a30329b326534faaae8fdffc7be911d1169b3869c081de612838caa6b"
):
    raise RuntimeError("Frozen plan SHA256 mismatch.")
plan = json.loads(plan_bytes)

progress_path = (
    repo / "docs/research_reports/notebook05b_screening_v1/"
    "progress_01579.json"
)
progress_bytes = progress_path.read_bytes()
if sha256(progress_bytes) != (
    "f131052e115ca584fddda89a445349e014c7513b5f87e30b7bb07865904ce801"
):
    raise RuntimeError("Checkpoint summary SHA256 mismatch.")
progress = json.loads(progress_bytes)
fingerprint = progress["session_fingerprint"]

baseline, baseline_sha = read_record(output_dir / "clean_evaluation.json.gz")
if (
    baseline["kind"] != "baseline"
    or baseline["session_fingerprint"] != fingerprint
    or baseline["plan_sha256"] != plan_sha
):
    raise RuntimeError("Baseline provenance mismatch.")

expected_scenarios = plan["scenarios"][:1579]
expected_files = {
    f"{row['scenario_id']}_evaluation.json.gz"
    for row in expected_scenarios
}
actual_files = {
    path.name for path in output_dir.glob("S05B-*_evaluation.json.gz")
}
if actual_files != expected_files:
    raise RuntimeError("Evaluation file set differs from this checkpoint.")
if list(output_dir.rglob("*.part")) or list(output_dir.glob("*_failure.json.gz")):
    raise RuntimeError("Partial or failure records require review.")

metric_names = list(baseline["metrics"])
rows = []
inventory = []

for number, scenario in enumerate(expected_scenarios, start=1):
    scenario_id = scenario["scenario_id"]
    evaluation_path = output_dir / f"{scenario_id}_evaluation.json.gz"
    prediction_path = output_dir / f"{scenario_id}_predictions.json.gz"
    record, evaluation_sha = read_record(evaluation_path)

    if (
        record["kind"] != "scenario_evaluation"
        or record["session_fingerprint"] != fingerprint
        or record["plan_sha256"] != plan_sha
        or record["scenario"] != scenario
        or record["clean_metrics"] != baseline["metrics"]
        or record["exact_restoration"] is not True
        or record["single_weight_isolation"] is not True
    ):
        raise RuntimeError(f"Scenario provenance mismatch: {scenario_id}")

    metrics = record["metrics"]
    if set(metrics) != set(metric_names):
        raise RuntimeError(f"Metric schema mismatch: {scenario_id}")
    if not all(math.isfinite(value) for value in metrics.values()):
        raise RuntimeError(f"Non-finite metrics: {scenario_id}")

    delta = metrics["AP50_95"] - baseline["metrics"]["AP50_95"]
    if (
        abs(record["AP50_95_change"] - delta) > 1e-12
        or abs(record["loss"] - max(0.0, -delta)) > 1e-12
    ):
        raise RuntimeError(f"AP arithmetic mismatch: {scenario_id}")

    if not prediction_path.is_file():
        raise FileNotFoundError(prediction_path)

    rows.append({
        "scenario_id": scenario_id,
        "layer": scenario["layer"],
        "bit": scenario["bit"],
        "replicate": scenario["replicate"],
        "flat_index": scenario["flat_index"],
        "q_clean": scenario["q_clean"],
        "q_faulty": scenario["q_faulty"],
        **metrics,
        "AP50_95_change": delta,
        "nonnegative_AP50_95_loss": record["loss"],
    })
    inventory.append({
        "scenario_id": scenario_id,
        "evaluation_sha256_verified_now": evaluation_sha,
        "prediction_path": prediction_path.relative_to(repo).as_posix(),
        "prediction_size_bytes": prediction_path.stat().st_size,
        "prediction_sha256_recorded_at_execution": record["prediction_sha256"],
        "prediction_checksum_rechecked_now": False,
    })

    if number % 200 == 0:
        print(f"Audited evaluation reports: {number}/1579")

stream = io.StringIO(newline="")
writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
writer.writeheader()
writer.writerows(rows)
csv_bytes = stream.getvalue().encode("utf-8")

audit = {
    "status": "PARTIAL_SCREENING_EVALUATION_AUDIT_COMPLETE",
    "completed_scenarios": len(rows),
    "planned_scenarios": plan["scenario_count"],
    "plan_sha256": plan_sha,
    "session_fingerprint": fingerprint,
    "progress_sha256": sha256(progress_bytes),
    "baseline_evaluation_sha256": baseline_sha,
    "clean_metrics": baseline["metrics"],
    "scenario_metrics_csv_sha256": sha256(csv_bytes),
    "verified": [
        "evaluation payload checksums",
        "scenario identity and plan provenance",
        "matched clean metrics",
        "recorded isolation and restoration flags",
        "AP change and nonnegative loss arithmetic",
        "prediction file presence",
    ],
    "limitations": [
        "Prediction contents and current checksums were not audited.",
        "Recorded restoration flags were inspected; inference was not rerun.",
        "Campaign is incomplete; no final layer ranking is claimed.",
    ],
    "artifacts": inventory,
}

files_to_write = {
    destination / "scenario_metrics_v1.csv": csv_bytes,
    destination / "evaluation_audit_v1.json": (
        json.dumps(audit, indent=2, allow_nan=False) + "\n"
    ).encode("utf-8"),
}

for path, content in files_to_write.items():
    if path.exists() and path.read_bytes() != content:
        raise RuntimeError(f"Existing publication file differs: {path.name}")

destination.mkdir(parents=True, exist_ok=True)
for path, content in files_to_write.items():
    if not path.exists():
        with path.open("xb") as handle:
            handle.write(content)
    if path.read_bytes() != content:
        raise RuntimeError(f"Write verification failed: {path.name}")
    print("Saved:", path)
    print("SHA256:", sha256(content))

print("Audited evaluation reports:", len(rows))
print("No inference. Original artifacts unchanged. No staging, commit or push.")

Audited evaluation reports: 200/1579
Audited evaluation reports: 400/1579
Audited evaluation reports: 600/1579
Audited evaluation reports: 800/1579
Audited evaluation reports: 1000/1579
Audited evaluation reports: 1200/1579
Audited evaluation reports: 1400/1579
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook05b_checkpoint_01579_v1/scenario_metrics_v1.csv
SHA256: d6412fec7d767d773786560734eccc8484c7d935a9dfe9cc74a49d45a3931b8b
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook05b_checkpoint_01579_v1/evaluation_audit_v1.json
SHA256: a7dee0776a260c0d4d0aab0809a1465931bcb92cec41bb8dc172dcdad4d94df6
Audited evaluation reports: 1579
No inference. Original artifacts unchanged. No staging, commit or push.


In [9]:
from pathlib import Path
import gzip
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
destination = (
    repo / "docs/research_reports/notebook05b_checkpoint_01579_v1"
)

def sha256(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":"),
    ).encode("utf-8")

audit_bytes = (destination / "evaluation_audit_v1.json").read_bytes()
if sha256(audit_bytes) != (
    "a7dee0776a260c0d4d0aab0809a1465931bcb92cec41bb8dc172dcdad4d94df6"
):
    raise RuntimeError("Evaluation audit SHA256 mismatch.")
audit = json.loads(audit_bytes)

sources = [
    (
        "results/notebook05b/int8_screening_v1/session.json.gz",
        "execution_session_v1.json",
        None,
    ),
    (
        "results/notebook05b/int8_screening_v1/clean_evaluation.json.gz",
        "screening_clean_evaluation_v1.json",
        audit["baseline_evaluation_sha256"],
    ),
    (
        "results/notebook05b/engineering_fault_validation_v1/validation_report.json.gz",
        "engineering_validation_v1.json",
        "bf6b03feece37b62ee686d0ddd0d6af685d7bf07d162b7833c2fe417da68ed06",
    ),
    (
        "results/notebook05b/shared_INT8_clean_gpu_timing_pilot50_v1.json",
        "clean_gpu_timing_v1.json",
        "3cbd9a01c0e525bd2c4601b119aec7039377c5ceb18d7eb60eb1932e6480af02",
    ),
]

prepared = {}

for relative_path, filename, expected_sha in sources:
    source = repo / relative_path
    source_bytes = source.read_bytes()
    source_sha = sha256(source_bytes)
    if expected_sha is not None and source_sha != expected_sha:
        raise RuntimeError(f"Source SHA256 mismatch: {source.name}")

    if source.suffix == ".gz":
        record = json.loads(gzip.decompress(source_bytes))
        payload = record["payload"]
        if sha256(canonical(payload)) != record["payload_sha256"]:
            raise RuntimeError(f"Payload checksum mismatch: {source.name}")
    else:
        record = json.loads(source_bytes)
        payload = record

    if filename == "execution_session_v1.json":
        if sha256(canonical(payload)) != audit["session_fingerprint"]:
            raise RuntimeError("Execution session fingerprint mismatch.")
        if payload["plan_sha256"] != audit["plan_sha256"]:
            raise RuntimeError("Execution session plan mismatch.")

    publication = {
        "source_relative_path": relative_path,
        "source_archive_sha256": source_sha,
        "record": record,
    }
    content = (
        json.dumps(publication, indent=2, allow_nan=False) + "\n"
    ).encode("utf-8")

    path = destination / filename
    if path.exists() and path.read_bytes() != content:
        raise RuntimeError(f"Existing publication file differs: {filename}")
    prepared[path] = content

for path, content in prepared.items():
    if not path.exists():
        with path.open("xb") as handle:
            handle.write(content)
    if path.read_bytes() != content:
        raise RuntimeError(f"Publication verification failed: {path.name}")
    print("Saved:", path.name, "| SHA256:", sha256(content))

print("\n--- Notebook 05b candidates ---")
for path in sorted(
    Path("/content/drive/MyDrive/Colab Notebooks").glob("05b*.ipynb")
):
    notebook = json.loads(path.read_bytes())
    print(path)
    print("Cells:", len(notebook.get("cells", [])))
    print("Size (bytes):", path.stat().st_size)

print("\nOriginal artifacts unchanged. No inference, staging, commit or push.")

Saved: execution_session_v1.json | SHA256: efc84a427e45aef0e7921bb69b70beb45fd3bb1b9b3d5c9fad4d967518205833
Saved: screening_clean_evaluation_v1.json | SHA256: 39c18cdf41849dddf97e1455883b835f47ff3a35d1d42b031c44b511c685a6a8
Saved: engineering_validation_v1.json | SHA256: 50fb2a6003dbebeca4599017a512374d57d9db0f28ea8021ebfe8bb45e87fe49
Saved: clean_gpu_timing_v1.json | SHA256: e9d62c0d52ef8c66b623c7ecd555d1478b014a0534488f1a4398cc54d5292d45

--- Notebook 05b candidates ---
/content/drive/MyDrive/Colab Notebooks/05b_fault_screening_design.ipynb
Cells: 28
Size (bytes): 234697

Original artifacts unchanged. No inference, staging, commit or push.
